# UAS Pembelajaran Mesin — Regression: Air Quality Dataset

**Universitas MDP** | EL4010 Pembelajaran Mesin | Model Validation and Generalization

| Item | Detail |
|------|--------|
| Kelas | 25/26/2/TE6A/EL4010 |
| Tugas | Regression — Dataset No. 1 |
| Target | `CO(GT)` — konsentrasi CO aktual (mg/m³) |
| Split | Train 60% · Validation 20% · Test 20% |
| GitHub | [Arcadiavr/uas-air-quality-regression](https://github.com/Arcadiavr/uas-air-quality-regression) |

---

## Link Pengumpulan (GitHub / Colab — akses view)

| Platform | Link |
|----------|------|
| **GitHub Repository** | https://github.com/Arcadiavr/uas-air-quality-regression |
| **Google Colab (view & run)** | https://colab.research.google.com/github/Arcadiavr/uas-air-quality-regression/blob/main/Air_Quality_UAS.ipynb |
| **Notebook di GitHub** | https://github.com/Arcadiavr/uas-air-quality-regression/blob/main/Air_Quality_UAS.ipynb |

---

## Link Dataset

| Sumber | Link |
|--------|------|
| UCI — Air Quality | https://archive.ics.uci.edu/dataset/360/air+quality |
| Download ZIP | https://archive.ics.uci.edu/static/public/360/air+quality.zip |
| File di repo | `data/AirQualityUCI.csv` |

---

## Referensi Paper

1. De Vito, S., et al. (2008). *On field calibration of an electronic nose for benzene estimation in an urban pollution monitoring scenario.* Sensors and Actuators B: Chemical. https://doi.org/10.1016/j.snb.2008.01.035
2. UCI ML Repository (2016). *Air Quality Data Set.* https://archive.ics.uci.edu/dataset/360/air+quality

## Referensi Code

- scikit-learn Regression: https://scikit-learn.org/stable/supervised_learning.html#regression
- `train_test_split`: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- `RandomForestRegressor`: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
- Regression metrics: https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics

---

## Pipeline ML
1. **Task Conceptualization** — definisi masalah & metrik
2. **Data Preparation** — cleaning, split train/val/test
3. **Model Training & Selection** — bandingkan model di validation set
4. **Final Evaluation** — evaluasi akhir di test set (ujian)

## Tahap 1 — Task Conceptualization

**Rumusan masalah:** Membangun model regresi untuk memprediksi konsentrasi karbon monoksida (CO) berdasarkan respons sensor gas dan kondisi lingkungan (suhu, kelembaban).

**Analogi dari slide:**
- **Training** = belajar dari buku (data latih)
- **Validation** = latihan soal (pilih model terbaik)
- **Testing** = ujian akhir (evaluasi final, dipakai sekali)

**Metrik evaluasi:** RMSE, MAE, R²

In [ ]:
# Setup environment (Colab + download dataset otomatis)
import subprocess
import urllib.request
import zipfile
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
    subprocess.run(["pip", "install", "-q", "seaborn"], check=False)
except ImportError:
    IN_COLAB = False

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("data")
DATA_PATH = DATA_DIR / "AirQualityUCI.csv"
UCI_ZIP_URL = "https://archive.ics.uci.edu/static/public/360/air+quality.zip"

if not DATA_PATH.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DATA_DIR / "air_quality.zip"
    print("Mengunduh dataset dari UCI...")
    urllib.request.urlretrieve(UCI_ZIP_URL, zip_path)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_DIR)
    print(f"Dataset diunduh ke: {DATA_PATH}")
else:
    print(f"Dataset lokal: {DATA_PATH}")

TARGET = "CO(GT)"
FEATURES = [
    "PT08.S1(CO)", "PT08.S2(NMHC)", "PT08.S3(NOx)", "PT08.S4(NO2)",
    "PT08.S5(O3)", "T", "RH", "AH",
]

sns.set_theme(style="whitegrid")
print(f"Environment: {'Google Colab' if IN_COLAB else 'Lokal'} | Library siap.")

## Tahap 2 — Data Collection & Preparation

Langkah preprocessing:
1. Load CSV (separator `;`, desimal `,`)
2. Ganti missing value `-200` dengan `NaN`
3. Hapus baris tanpa target
4. Split **Train 60% / Validation 20% / Test 20%**

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";", decimal=",", na_values=["-200", -200])
df.columns = df.columns.str.strip()
df = df.drop(columns=[c for c in df.columns if c == "" or c.startswith("Unnamed")], errors="ignore")

for col in FEATURES + [TARGET]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df_clean = df.dropna(subset=[TARGET]).copy()
print(f"Baris awal: {len(df)}  ->  setelah cleaning: {len(df_clean)}")
df_clean[FEATURES + [TARGET]].describe().round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(df_clean[FEATURES + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax)
ax.set_title("Korelasi Fitur dan Target CO(GT)")
plt.show()

In [ ]:
X = df_clean[FEATURES]
y = df_clean[TARGET]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE)

print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")

## Tahap 3 — Model Training & Selection

Latih 5 model, evaluasi di **validation set**, pilih yang RMSE-nya terendah.

> **Penting:** Test set belum disentuh di tahap ini.

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
    }

base = [("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]
models = {
    "Linear Regression": Pipeline(base + [("model", LinearRegression())]),
    "Ridge": Pipeline(base + [("model", Ridge(alpha=1.0))]),
    "Lasso": Pipeline(base + [("model", Lasso(alpha=0.01, max_iter=10000))]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", GradientBoostingRegressor(random_state=RANDOM_STATE)),
    ]),
}

results = []
fitted = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted[name] = model
    val_m = evaluate(y_val, model.predict(X_val))
    test_m = evaluate(y_test, model.predict(X_test))
    results.append({"Model": name, **{f"{k}_Val": v for k, v in val_m.items()}, **{f"{k}_Test": v for k, v in test_m.items()}})

comparison = pd.DataFrame(results).sort_values("RMSE_Val").reset_index(drop=True)
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(comparison["Model"], comparison["RMSE_Val"], color="steelblue")
axes[0].set_title("RMSE — Validation Set")
axes[0].set_ylabel("RMSE (mg/m³)")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(comparison["Model"], comparison["R2_Val"], color="seagreen")
axes[1].set_title("R² — Validation Set")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

best_name = comparison.iloc[0]["Model"]
print(f"Model terpilih: {best_name}")

## Tahap 4 — Final Evaluation (Test Set)

Evaluasi akhir pada **test set** — analogi "ujian akhir" dari slide.

Cek **generalisasi**: apakah performa validation dan test konsisten?

In [ ]:
best = comparison.iloc[0]
y_pred_test = fitted[best_name].predict(X_test)

print("=== HASIL FINAL (TEST SET) ===")
print(f"Model     : {best['Model']}")
print(f"RMSE      : {best['RMSE_Test']:.4f} mg/m³")
print(f"MAE       : {best['MAE_Test']:.4f} mg/m³")
print(f"R²        : {best['R2_Test']:.4f}")
print(f"Selisih RMSE Val-Test: {abs(best['RMSE_Val'] - best['RMSE_Test']):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred_test, alpha=0.4)
lims = [min(y_test.min(), y_pred_test.min()), max(y_test.max(), y_pred_test.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Aktual CO(GT)")
axes[0].set_ylabel("Prediksi CO(GT)")
axes[0].set_title(f"Actual vs Predicted — {best_name}")

residuals = y_test - y_pred_test
axes[1].scatter(y_pred_test, residuals, alpha=0.4)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Prediksi CO(GT)")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual Plot")

plt.tight_layout()
plt.show()

## Kesimpulan

1. Dataset Air Quality berisi respons sensor gas di kota Italia (2004–2005).
2. Setelah menghapus missing value (`-200`), tersisa **7.674** sampel valid.
3. **Random Forest** terbaik di validation set (RMSE terendah).
4. Di test set, model mencapai **R² ≈ 0.86** — prediksi cukup akurat.
5. Selisih RMSE validation vs test kecil → model **tidak overfitting berat** (generalisasi baik).

### Keterbatasan
- Sensor mengalami drift seiring waktu (concept drift).
- Missing value cukup banyak di beberapa fitur.
- Model belum diuji pada kondisi lapangan baru.

### Saran
- Tambah feature engineering waktu (jam, bulan).
- Coba cross-validation untuk validasi lebih robust.
- Bandingkan dengan model regresi yang lebih sederhana sebagai baseline.

## Lampiran Referensi (untuk laporan)

### Link Pengumpulan
- GitHub: https://github.com/Arcadiavr/uas-air-quality-regression
- Colab: https://colab.research.google.com/github/Arcadiavr/uas-air-quality-regression/blob/main/Air_Quality_UAS.ipynb

### Dataset
- UCI Air Quality: https://archive.ics.uci.edu/dataset/360/air+quality
- Download ZIP: https://archive.ics.uci.edu/static/public/360/air+quality.zip

### Paper
- De Vito, S., et al. (2008). *Sensors and Actuators B: Chemical*. https://doi.org/10.1016/j.snb.2008.01.035

### Code / Library
| Library | Link |
|---------|------|
| scikit-learn Regression | https://scikit-learn.org/stable/supervised_learning.html#regression |
| `train_test_split` | https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html |
| `RandomForestRegressor` | https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html |
| Regression metrics | https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics |

Lihat `SUBMISSION_LINKS.md` untuk template lengkap laporan.